In [2]:
import librosa
import soundfile as sf
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import torch
import os
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import matplotlib as plt

load_dotenv()
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token)

# audio to numpy
#audio, sr = librosa.load("Stimuli\Raw_resampled\dash-tash_F0_11_VOT_11.wav", sr=16000)
#sf.write("dash-tash_11_resampled.wav", audio, 16000)

# load a soundfile in 16kHz
dash, sr = librosa.load("Stimuli\Continua\dash-tash\dash-tash_F0_07_VOT_07.wav", sr=16000)

ImportError: DLL load failed while importing _imaging: The specified module could not be found.

In [3]:
# preprocess numpy audio to tensors
pre_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
dash_tensor = pre_processor(dash, sampling_rate=sr, return_tensors='pt')

print(dash_tensor)
print(dash_tensor['input_values'].shape)

{'input_values': tensor([[0.0023, 0.0023, 0.0023,  ..., 0.0065, 0.0094, 0.0023]])}
torch.Size([1, 19831])


In [5]:
# run CNN and 12-layer transformer
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

with torch.no_grad():
    embeddings = model(**dash_tensor, output_hidden_states=True)


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
from transformers import Wav2Vec2ForCTC
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

logits = model(**dash_tensor).logits
 
# take argmax and decode
predicted_ids = torch.argmax(logits, dim=-1)
transcription = pre_processor.batch_decode(predicted_ids)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
print(transcription)

['TASHE']


In [54]:
transcripts = []
steps = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11"]
#continua = 

#path = os.path.join("Stimuli", "Continua", "dash", "dash-tash_F0_01_VOT_01.wav")

for step in steps:
    continuum_step, sr = librosa.load(f"Stimuli/Continua/dash-tash/dash-tash_F0_{step}_VOT_{step}.wav", sr=16000)
    
    tensor = pre_processor(continuum_step, sampling_rate=sr, return_tensors='pt')
    logits = model(**tensor).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = pre_processor.batch_decode(predicted_ids)
    transcripts.append(transcription)

print(transcripts)

Task was destroyed but it is pending!
task: <Task pending name='Task-238' coro=<_async_in_context.<locals>.run_in_context_pre311() done, defined at C:\Users\HP\miniconda3\envs\thesis\lib\site-packages\ipykernel\utils.py:76> wait_for=<Task pending name='Task-239' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\HP\miniconda3\envs\thesis\lib\site-packages\ipykernel\utils.py:68> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\HP\miniconda3\envs\thesis\lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\HP\miniconda3\envs\thesis\lib\site-packages\torch\nn\functional.py:1295: RuntimeWarning: coroutine '_async_in_context.<locals>.preserve_context' was never awaited
  return _VF.dropout_(input, p, training) if inplace else _VF.dropout(input, p, training)
Task was destroyed but it is pending!
task: <Task pending name='Task-239' coro=<_async_in_context.<locals>.preserve_context() running at C:\Users\HP\miniconda3\envs\thesis\

[['GAS'], ['DASH'], ['DASH'], ['DASH'], ['DASH'], ['TAS'], ['TASHE'], ['TASH'], ['TASH'], ['TASHE'], ['TASHE']]


In [4]:
# output of the first hidden layer (CNN)
print(embeddings.hidden_states[0])

print(embeddings.hidden_states[0].shape)

tensor([[[-0.1989, -0.1269, -0.3311,  ...,  0.1660,  0.5340,  0.1340],
         [ 0.0525,  0.3866, -0.2554,  ...,  0.8195,  0.4377, -0.2764],
         [ 0.5461,  0.1322, -0.0841,  ..., -0.2403, -0.2208, -0.8773],
         ...,
         [ 0.4066,  0.0362,  0.0397,  ..., -0.0381,  0.3028,  0.1629],
         [ 0.4060,  0.0885,  0.0225,  ...,  0.0708,  0.5041,  0.2288],
         [ 0.5137, -0.0737,  0.2317,  ...,  0.0112,  0.5276,  0.2689]]])
torch.Size([1, 60, 768])


In [57]:
start_frame = int(0.05 // 0.02)
end_frame = int(0.43 // 0.02)

# take hidden representations for a specific layer and time window, then average them into a single vector
mean = embeddings.hidden_states[0][0, start_frame:end_frame +1, :].mean(dim=0)
print(mean.shape)

# should be 13 (CNN + 12 transformer layers)
print(len(embeddings.hidden_states))

# CNN output shape
print(embeddings.hidden_states[0].shape)

torch.Size([768])
13
torch.Size([1, 61, 768])


In [62]:
dash_embed = {}

steps = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11"]

dash_embed["0"] = {}
dash_embed["0"]["01"] = mean.numpy()

print(dash_embed)



{'0': {'01': array([ 4.04970571e-02, -2.24632859e-01, -1.12786219e-01,  9.23789740e-02,
        7.57273957e-02, -1.27615780e-01,  2.27058798e-01,  3.45115848e-02,
        1.24639593e-01,  6.86646104e-02, -9.92152542e-02, -1.45098209e-01,
       -1.08445939e-02,  6.34466037e-02, -2.16369033e-02, -2.13714078e-01,
       -1.25452310e-01, -9.23581868e-02,  2.07136534e-02, -2.51984298e-02,
       -1.52058110e-01, -1.37401164e-01,  5.06347418e-01, -3.90252881e-02,
        2.07822789e-02,  1.17519442e-02,  4.10907328e-01,  7.77502507e-02,
       -6.59432560e-02,  3.84098142e-02, -1.05480030e-01,  9.87870321e-02,
       -1.77623574e-02, -5.81946857e-02, -9.32639688e-02,  2.01642349e-01,
        1.00210965e-01,  2.17960775e-02,  6.43205643e-03,  2.12730709e-02,
        8.55044574e-02, -1.84374787e-02, -7.04548582e-02, -3.31932381e-02,
       -1.38917565e-01, -2.19409401e-03, -9.94759649e-02,  1.20844804e-01,
       -1.76672488e-02,  1.68522060e-01,  8.01627487e-02,  1.27905220e-01,
       -1.06

In [6]:

def get_continuum_embeddings(sound_name):
    start_frame = int(0.05 // 0.02)
    steps = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11"]
    embed_dict = {}

    for layer in range(13):
        embed_dict[layer] = {}
    
    for i, step in enumerate(steps):
        # load and preprocess
        sound, sr = librosa.load(f"Stimuli/Continua/{sound_name}/{sound_name}_F0_{step}_VOT_{step}.wav", sr=16000)
        sound_tensor = pre_processor(sound, sampling_rate=sr, return_tensors='pt')

        # max: prevent an error for the first frame where VOT = 0
        end_frame = max(int((0.05 + i * 0.009) // 0.02), start_frame + 1)
    
        # run wav2vec2
        with torch.no_grad():
            embeddings = model(**sound_tensor, output_hidden_states=True)
    
        # extract embeddings of each layer
        for layer in range(13):
            mean = embeddings.hidden_states[layer][0, start_frame:end_frame, :].mean(dim=0)
            embed_dict[layer][step] = mean.numpy()

    return embed_dict

dash_tash_embed = get_continuum_embeddings("dash-tash")
task_dask_embed = get_continuum_embeddings("task-dask")


In [8]:

def cosine_distance(x, endpoint):
    dot_product = np.dot(x, endpoint)
    norm_x = np.linalg.norm(x)
    norm_endpoint = np.linalg.norm(endpoint)
    
    return 1 - (dot_product / (norm_x * norm_endpoint))

def cosine_sim(x, t, d):
    cos_dist_x_t = cosine_distance(x, t)
    cos_dist_x_d = cosine_distance(x, d)
    sim_x_t = 1 - (cos_dist_x_t / (cos_dist_x_t + cos_dist_x_d))
    
    return sim_x_t

def get_embed_sim(embed_dict_input):
    similarities = {}
    steps = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11"]
    
    for layer in range(13):
        t = embed_dict_input[layer]["11"]
        d = embed_dict_input[layer]["01"]
        similarities[layer] = []
        
        for step in steps:
            embed_sim = cosine_sim(embed_dict_input[layer][step], t, d)
            similarities[layer].append(embed_sim)
            
    return similarities

results = get_embed_sim(dash_tash_embed)

In [103]:
print(results)

{0: [np.float32(0.0), np.float32(0.15084845), np.float32(0.2185629), np.float32(0.2815079), np.float32(0.3962816), np.float32(0.50202286), np.float32(0.56756425), np.float32(0.64330745), np.float32(0.7687148), np.float32(0.7871129), np.float32(1.0)], 1: [np.float32(0.0), np.float32(0.16223508), np.float32(0.22741216), np.float32(0.33480072), np.float32(0.4598866), np.float32(0.4815678), np.float32(0.5798831), np.float32(0.6319132), np.float32(0.7096651), np.float32(0.81686103), np.float32(1.0)], 2: [np.float32(-1.1920929e-06), np.float32(0.124022245), np.float32(0.17424172), np.float32(0.35722995), np.float32(0.4775831), np.float32(0.46984172), np.float32(0.54377425), np.float32(0.6593793), np.float32(0.76030034), np.float32(0.8322315), np.float32(1.0000012)], 3: [np.float32(-1.0728836e-06), np.float32(0.114109814), np.float32(0.17540336), np.float32(0.34508592), np.float32(0.4684199), np.float32(0.51082814), np.float32(0.58913094), np.float32(0.69429), np.float32(0.7986684), np.float3

In [9]:
import seaborn as sns

y = results[0]

sns.lineplot(y)
plt.xlabel("Continuum step")
plt.ylabel("Similarity to /t/")
plt.title("CNN layer")
plt.show()


ImportError: DLL load failed while importing _imaging: The specified module could not be found.

In [93]:
# some checks
print(dash_tash_embed[4]["11"].shape)
print(dash_tash_embed.keys())
print(dash_tash_embed[0].keys())

print(task_dask_embed[3]["03"].shape)
print(task_dask_embed.keys())
print(task_dask_embed[0].keys())

(768,)
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
dict_keys(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11'])
(768,)
dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
dict_keys(['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11'])


In [ ]:
for step in steps:
    continuum_step, sr = librosa.load(f"Stimuli/Continua/dash-tash/dash-tash_F0_{step}_VOT_{step}.wav", sr=16000)
    
    tensor = pre_processor(continuum_step, sampling_rate=sr, return_tensors='pt')
    logits = model(**tensor).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = pre_processor.batch_decode(predicted_ids)
    transcripts.append(transcription)